In [1]:
import os
import sys
import shutil
import time

import numpy as np
from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'serif'
plt.rcParams['lines.linewidth'] = 1.25


# import dustpy as dp
from dustpy import Simulation
from dustpy import constants as c


print("Done")



A newer version of DustPy is available.
This version:   1.0.5
Latest version: 1.0.8

Upgrade with
pip install dustpy --upgrade

Done


In [2]:
def convert(S, m):
    """Function converts the Dustpy units into number densities.

    Parameters
    ----------
    S : array
        Integrated surface density in DustPy units
    m : array
        mass grid

    Returns
    -------
    Nm2 : array
        Simulation results in desired units for comparison"""
    A = np.mean(m[1:]/m[:-1])
    B = 2 * (A-1) / (A+1)
    return S / B

def solution_constant_kernel(t, m, a, S0):
    """Analytical solution of the constant collision kernel R(m, m') = a.
    Initial condition is that only the zeroth mass bin is filled.

    Parameters
    ----------
    t : float
        Time
    m : Field
        Mass grid
    a : float
        Kernel constant
    S0 : float
        Total dust surface density

    Returns
    -------
    Nm2 : Field
        Analytical solution in desired units."""
    m0 = m[0]
    N0 = S0 / m0
    return N0 / m0 * 4./(a*N0*t)**2 * np.exp( (1.-m/m0) * 2/(a*N0*t) ) * m**2


def setup_simulation(sim, S0):
    # Turning off gas evolution by removing integrator instruction
    del(sim.integrator.instructions[1])
    # Turning off gas source to not influence the time stepping
    sim.gas.S.tot[...] = 0.
    sim.gas.S.tot.updater = None
    # Turning off dust advection
    sim.dust.v.rad[...] = 0.
    sim.dust.v.rad.updater = None
    
    # Setting the initial time
    sim.t = 1.e-9
    # Setting the initial dust surface density
    m = sim.grid.m
    A = np.mean(m[1:]/m[:-1])
    B = 2 * (A-1) / (A+1)
    sim.dust.Sigma[...] = sim.dust.SigmaFloor[...]
    sim.dust.Sigma[1, :] = np.maximum(solution_constant_kernel(sim.t, m, 1., S0)*B, sim.dust.SigmaFloor[1, :])
    # Updating the simulation object
    sim.update()

In [3]:
# create simulation
sim = Simulation()

# change ini params and initialize simulation
sim.ini.dust.allowDriftingParticles = False
sim.ini.grid.Nr = 3                             # 0D with 2 ghost cells for boundary conditions
sim.ini.grid.Nmbpd = 7
S0 = 1.

sim.initialize()
setup_simulation(sim, S0)

snapshots = np.logspace(-7.,3.,20)
sim.t.snapshots = snapshots


In [49]:
# sim.ini.grid
# sim.ini.dust

# sim.grid.m.size # 120
# sim.dust.v.rel.tot.size #43200 = 120 * 120 * 3
# sim.dust.v.rel.tot = 0